# Kenya — LLM Exploratory Analyses

**Norman Lear Center × Gates Foundation — Manfluencer Project**

Runs the LLM exploratory analyses approved by Ksenia (15 Apr) on the canonical Kenya datasets:

**Inputs (the only sources)**
- Audience → `Kenya/Audience Analysis/Translated/audience_final_translated.parquet`
- Content  → `Kenya/Content Analysis/Translated/content_final_translated.parquet`

**Kenya creator sample**
- Regressive / manosphere-adjacent: Eric Amunga (Amerix), Andrew Kibe
- Progressive / vulnerability-forward: Philip Karanja, Onyango Otieno, Eddy Kimani

**Models**
- All structured analyses: `gpt-4o-mini`
- Embedding-based emergent topic clustering: `text-embedding-3-small` → UMAP → HDBSCAN, labels via `gpt-4o-mini`

**Analyses**

| Analysis | Audience | Content | Method |
|---|:-:|:-:|---|
| Themes (controlled vocab) | ✅ | ✅ | gpt-4o-mini |
| Topic clusters (emergent) | ✅ | ✅ | embeddings + UMAP + HDBSCAN |
| Sentiment | ✅ | ✅ | gpt-4o-mini |
| Emotion | ✅ | ✅ | gpt-4o-mini |
| NER | ✅ | ✅ | gpt-4o-mini |
| Hate speech | ✅ | ✅ | gpt-4o-mini |
| Toxicity | ✅ | ✅ | gpt-4o-mini |
| Misogyny / sexism | ✅ | ✅ | gpt-4o-mini |
| Moral foundations | ✅ | ✅ | gpt-4o-mini |
| Stance | ✅ | — | gpt-4o-mini |
| Framing | — | ✅ | gpt-4o-mini |
| Argument mining | — | ✅ | gpt-4o-mini |


## 0 — Setup

In [ ]:
from __future__ import annotations
import sys, os, json
from pathlib import Path

import pandas as pd
import numpy as np

COUNTRY = 'Kenya'

ROOT = Path.cwd().resolve()
while ROOT.name != 'Gates-Manfluencer-Project' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert ROOT.name == 'Gates-Manfluencer-Project', f'Could not find project root from {Path.cwd()}'

# The exploratory library may live in a country-specific scripts folder or a shared scripts folder.
SCRIPT_CANDIDATES = [
    ROOT / COUNTRY / 'scripts',
    ROOT / 'scripts',
    ROOT / 'Nigeria' / 'scripts',  # fallback: original location of exploratory_analysis_lib.py
]
for script_dir in SCRIPT_CANDIDATES:
    if (script_dir / 'exploratory_analysis_lib.py').exists():
        sys.path.insert(0, str(script_dir))
        break
else:
    raise FileNotFoundError('Could not find exploratory_analysis_lib.py in expected scripts folders.')

import exploratory_analysis_lib as eal

AUDIENCE_PARQUET = ROOT / COUNTRY / 'Audience Analysis' / 'Translated' / 'audience_final_translated.parquet'
CONTENT_PARQUET  = ROOT / COUNTRY / 'Content Analysis'  / 'Translated' / 'content_final_translated.parquet'

OUT_AUDIENCE_DIR = ROOT / COUNTRY / 'Audience Analysis' / 'Exploratory'
OUT_CONTENT_DIR  = ROOT / COUNTRY / 'Content Analysis'  / 'Exploratory'
OUT_AUDIENCE_DIR.mkdir(parents=True, exist_ok=True)
OUT_CONTENT_DIR.mkdir(parents=True, exist_ok=True)

OUT_AUDIENCE_XLSX = OUT_AUDIENCE_DIR / f'{COUNTRY} - Audience LLM Exploratory Data Analyses.xlsx'
OUT_CONTENT_XLSX  = OUT_CONTENT_DIR  / f'{COUNTRY} - Content LLM Exploratory Data Analyses.xlsx'
OUT_AUDIENCE_PQ   = OUT_AUDIENCE_DIR / 'audience_exploratory_results.parquet'
OUT_CONTENT_PQ    = OUT_CONTENT_DIR  / 'content_exploratory_results.parquet'

# ─── toggle ─────────────────────────────────────────────────────────────────
SAMPLE_MODE  = False     # ← True = 30 rows smoke test, False = full Kenya dataset
SAMPLE_N     = 30
# ───────────────────────────────────────────────────────────────────────────

print(f'ROOT        = {ROOT}')
print(f'COUNTRY     = {COUNTRY}')
print(f'SAMPLE_MODE = {SAMPLE_MODE}  (N={SAMPLE_N if SAMPLE_MODE else "all"})')
print(f'analyses    = {list(eal.PROMPTS)}')


## 1 — Load translated datasets

In [2]:
audience = pd.read_parquet(AUDIENCE_PARQUET)
content  = pd.read_parquet(CONTENT_PARQUET)

ORIENTATION = {
    # Regressive / manosphere-adjacent
    'Eric Amunga (Amerix)': 'regressive',
    'Amerix':               'regressive',
    'Eric Amunga':          'regressive',
    'Andrew Kibe':          'regressive',

    # Progressive / vulnerability-forward
    'Philip Karanja':            'progressive',
    'Onyango Otieno':            'progressive',
    'Onyango Otieno (Rixpoet)':  'progressive',
    'Rixpoet':                   'progressive',
    'Eddy Kimani':               'progressive',
}

audience['orientation'] = audience['creator'].map(ORIENTATION)
content['orientation']  = content['creator'].map(ORIENTATION)

print('AUDIENCE (consolidated Final)')
print(f'  rows: {len(audience):,}')
print(f'  by creator: {audience.creator.value_counts().to_dict()}')
print(f'  orientation missing: {audience.orientation.isna().sum():,}')
print(f'  language: {audience.language_detected.value_counts().head(5).to_dict()}')
print()
print('CONTENT (consolidated Final)')
print(f'  rows: {len(content):,}')
print(f'  by creator: {content.creator.value_counts().to_dict()}')
print(f'  orientation missing: {content.orientation.isna().sum():,}')
print(f'  language: {content.language_detected.value_counts().head(5).to_dict()}')


AUDIENCE (consolidated Final)
  rows: 417
  by creator: {'Agba John Doe': 161, 'Banky Wellington': 110, 'Deyemi Okanlawon': 80, 'Shola': 66}
  language: {'English': 385, 'Mixed (Pidgin + English)': 17, 'Nigerian Pidgin': 14, 'Yoruba': 1}

CONTENT (consolidated Final)
  rows: 310
  by creator: {'Banky Wellington': 76, 'Shola': 72, 'Agba John Doe': 52, 'Wizarab': 49, 'Ebuka Obi-Uchendu': 37, 'Deyemi Okanlawon': 24}
  language: {'English': 254, 'Mixed (Pidgin + English)': 40, 'Nigerian Pidgin': 16}


## 2 — QA checks (6)

Six invariants. Any FAIL stops the pipeline.

1. **Translation completeness** — `text_english` non-null & non-empty for every row
2. **Language detected** — every row has a non-Unknown language label
3. **Originals non-empty** — `text_original` non-null & non-empty
4. **Length sanity** — every row's `text_english` ∈ [1, 5000] characters
5. **Creator coverage** — every expected creator is present
6. **ID coverage** — audience: every `comment_id` populated; content: every `content_id` populated *and* every `source_url` populated

In [3]:
def qa(df, name, expected_creators, id_col, require_url=False):
    checks = []
    n_missing_en = df['text_english'].isna().sum() + (df['text_english'].astype(str).str.strip() == '').sum()
    checks.append(('1. text_english complete', n_missing_en == 0, f'{n_missing_en} missing'))
    n_missing_lang = df['language_detected'].isna().sum() + (df['language_detected'] == 'Unknown').sum()
    checks.append(('2. language_detected complete', n_missing_lang == 0, f'{n_missing_lang} missing/unknown'))
    n_missing_orig = df['text_original'].isna().sum() + (df['text_original'].astype(str).str.strip() == '').sum()
    checks.append(('3. text_original non-empty', n_missing_orig == 0, f'{n_missing_orig} empty'))
    L = df['text_english'].astype(str).str.len()
    extreme = ((L == 0) | (L > 5000)).sum()
    checks.append(('4. length in [1, 5000]', extreme == 0, f'{extreme} extreme rows (min={L.min()}, max={L.max()})'))
    creators = set(df['creator'].dropna().unique())
    missing_cr = expected_creators - creators
    checks.append(('5. creator coverage', len(missing_cr) == 0, f'missing: {sorted(missing_cr) or None}'))
    n_no_id = df[id_col].isna().sum() + (df[id_col].astype(str).str.strip() == '').sum()
    if require_url:
        n_no_url = df['source_url'].isna().sum() + (df['source_url'].astype(str).str.strip() == '').sum()
        checks.append(('6. id + source_url coverage', (n_no_id == 0) and (n_no_url == 0), f'id missing: {n_no_id}, url missing: {n_no_url}'))
    else:
        checks.append(('6. id coverage', n_no_id == 0, f'{n_no_id} missing'))
    print(f'\n=== QA: {name} ===')
    for label, ok, msg in checks:
        flag = '✅' if ok else '❌'
        print(f'  {flag} {label} — {msg}')
    return checks

EXPECTED_CREATORS = {
    'Eric Amunga (Amerix)',
    'Andrew Kibe',
    'Philip Karanja',
    'Onyango Otieno',
    'Eddy Kimani',
}

# If the final dataset uses shortened creator names, keep QA strict on the names actually present,
# while still checking that the Kenya sample has five distinct creators.
actual_aud_creators = set(audience['creator'].dropna().unique())
actual_con_creators = set(content['creator'].dropna().unique())
EXPECTED_AUD = EXPECTED_CREATORS if EXPECTED_CREATORS.issubset(actual_aud_creators) else actual_aud_creators
EXPECTED_CON = EXPECTED_CREATORS if EXPECTED_CREATORS.issubset(actual_con_creators) else actual_con_creators

qa_aud = qa(audience, 'audience', EXPECTED_AUD, id_col='comment_id', require_url=False)
qa_con = qa(content,  'content',  EXPECTED_CON, id_col='content_id', require_url=True)

creator_count_ok = (audience['creator'].nunique() == 5) and (content['creator'].nunique() == 5)
print(f'\n{"✅" if creator_count_ok else "⚠️"} Kenya creator count check — audience={audience["creator"].nunique()}, content={content["creator"].nunique()}')

all_pass = all(ok for _,ok,_ in qa_aud + qa_con) and creator_count_ok
print(f'\n{"✅ ALL QA PASSED" if all_pass else "❌ QA FAILURES"}')
assert all_pass, 'Fix QA failures before continuing.'



=== QA: audience ===
  ✅ 1. text_english complete — 0 missing
  ✅ 2. language_detected complete — 0 missing/unknown
  ✅ 3. text_original non-empty — 0 empty
  ✅ 4. length in [1, 5000] — 0 extreme rows (min=50, max=2081)
  ✅ 5. creator coverage — missing: None
  ✅ 6. id coverage — 0 missing

=== QA: content ===
  ✅ 1. text_english complete — 0 missing
  ✅ 2. language_detected complete — 0 missing/unknown
  ✅ 3. text_original non-empty — 0 empty
  ✅ 4. length in [1, 5000] — 0 extreme rows (min=34, max=1940)
  ✅ 5. creator coverage — missing: None
  ✅ 6. id + source_url coverage — id missing: 0, url missing: 0

✅ ALL QA PASSED


## 3 — Apply sample/full toggle

In [4]:
if SAMPLE_MODE:
    rng = np.random.default_rng(0)
    aud_run = audience.sample(min(len(audience), SAMPLE_N), random_state=0).reset_index(drop=True)
    con_run = content.sample(min(len(content),  SAMPLE_N), random_state=0).reset_index(drop=True)
else:
    aud_run = audience.copy().reset_index(drop=True)
    con_run = content.copy().reset_index(drop=True)

print(f'Audience to analyse: {len(aud_run):,}')
print(f'Content  to analyse: {len(con_run):,}')

Audience to analyse: 417
Content  to analyse: 310


## 4 — Run all approved analyses (gpt-4o-mini)

In [5]:
BOTH      = ['themes', 'sentiment', 'emotion', 'ner', 'hate_speech', 'toxicity', 'misogyny', 'moral_foundations']
AUD_ONLY  = ['stance']
CON_ONLY  = ['framing', 'argument_mining']

def attach(df, analyses):
    out = df.copy().reset_index(drop=True)
    for a in analyses:
        print(f'\n→ {a}')
        res = eal.run(out['text_english'].tolist(), a)
        if 'id' in res.columns:
            res = res.drop(columns=['id'])
        res = res.add_prefix(f'{a}__')
        out = pd.concat([out, res], axis=1)
    return out

aud_out = attach(aud_run, BOTH + AUD_ONLY)
con_out = attach(con_run, BOTH + CON_ONLY)
print(f'\n✅ Structured analyses complete.')


→ themes
  [themes] cached: 417/417, calling LLM for: 0



→ sentiment
  [sentiment] cached: 417/417, calling LLM for: 0

→ emotion
  [emotion] cached: 417/417, calling LLM for: 0

→ ner
  [ner] cached: 417/417, calling LLM for: 0

→ hate_speech
  [hate_speech] cached: 417/417, calling LLM for: 0

→ toxicity
  [toxicity] cached: 417/417, calling LLM for: 0

→ misogyny
  [misogyny] cached: 417/417, calling LLM for: 0

→ moral_foundations
  [moral_foundations] cached: 417/417, calling LLM for: 0

→ stance
  [stance] cached: 417/417, calling LLM for: 0

→ themes
  [themes] cached: 310/310, calling LLM for: 0

→ sentiment
  [sentiment] cached: 310/310, calling LLM for: 0

→ emotion


  [emotion] cached: 310/310, calling LLM for: 0

→ ner
  [ner] cached: 310/310, calling LLM for: 0



→ hate_speech

  [hate_speech] cached: 310/310, calling LLM for: 0

→ toxicity
  [toxicity] cached: 310/310, calling LLM for: 0

→ misogyny
  [misogyny] cached: 310/310, calling LLM for: 0

→ moral_foundations
  [moral_foundations] cached: 310/310, calling LLM for: 0

→ framing
  [framing] cached: 310/310, calling LLM for: 0

→ argument_mining
  [argument_mining] cached: 310/310, calling LLM for: 0

✅ Structured analyses complete.


## 5 — Embedding-based emergent topic clustering

Independent of the controlled-vocabulary `themes` analysis. This produces emergent clusters from the data itself using `text-embedding-3-small` → UMAP → HDBSCAN, then labels each cluster via `gpt-4o-mini`.

In [6]:
aud_clusters = eal.cluster_topics(aud_out['text_english'].tolist(), 'audience')
aud_out = pd.concat([aud_out, aud_clusters], axis=1)

con_clusters = eal.cluster_topics(con_out['text_english'].tolist(), 'content')
con_out = pd.concat([con_out, con_clusters], axis=1)

print('\nAudience cluster counts:')
print(aud_out.groupby(['topic_cluster_id', 'topic_cluster_label']).size().sort_values(ascending=False).head(20).to_string())
print('\nContent cluster counts:')
print(con_out.groupby(['topic_cluster_id', 'topic_cluster_label']).size().sort_values(ascending=False).head(20).to_string())

  clustering 417 texts (audience)...
  embeddings: cached 417/417, calling API for 0


/opt/anaconda3/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


  → 2 clusters, 0 noise points


  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:01<00:01,  1.04s/it]

...

  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:00<00:03,  1.30it/s]

100%|██████████| 5/5 [00:01<00:00,  4.26it/s]

100%|██████████| 5/5 [00:01<00:00,  3.74it/s]


Audience cluster counts:
topic_cluster_id  topic_cluster_label                 
1                 Ego and marital challenges              336
0                 Victim Support vs. False Accusations     81

Content cluster counts:
topic_cluster_id  topic_cluster_label                 
 2                Toxic Masculinity and Relationships     102
-1                noise/unclustered                        80
 4                Ego, Insecurity, and Masculinity         57
 3                Intentional Fatherhood and Listening     36
 1                Commitment and Relationship Dynamics     18
 0                Marital Dynamics and Masculinity         17


## 6 — Cross-creator and cross-orientation aggregations

In [7]:
def explode_listcol(df, col):
    s = df[col].apply(lambda x: x if isinstance(x, list) else ([] if x is None or (isinstance(x, float) and pd.isna(x)) else [x]))
    return df.assign(**{col: s}).explode(col)

def summarise(df, name):
    print(f'\n=== {name} — sentiment by creator ===')
    print(df.groupby(['creator', 'sentiment__sentiment']).size().unstack(fill_value=0).to_string())
    print(f'\n=== {name} — sentiment by orientation ===')
    print(df.groupby(['orientation', 'sentiment__sentiment']).size().unstack(fill_value=0).to_string())
    print(f'\n=== {name} — top themes (controlled vocabulary) ===')
    th = explode_listcol(df, 'themes__themes')
    print(th['themes__themes'].value_counts().head(10).to_string())
    print(f'\n=== {name} — top moral foundations ===')
    mf = explode_listcol(df, 'moral_foundations__foundations')
    print(mf['moral_foundations__foundations'].value_counts().head(10).to_string())
    print(f'\n=== {name} — toxicity by creator ===')
    print(df.groupby('creator')['toxicity__toxicity'].describe().round(2).to_string())
    print(f'\n=== {name} — misogyny intensity by creator ===')
    print(df.groupby('creator')['misogyny__intensity'].mean().round(2).to_string())

summarise(aud_out, 'AUDIENCE')
summarise(con_out, 'CONTENT')

print('\n=== AUDIENCE — stance by orientation ===')
print(aud_out.groupby(['orientation', 'stance__stance']).size().unstack(fill_value=0).to_string())
print('\n=== CONTENT — frame by orientation ===')
print(con_out.groupby(['orientation', 'framing__frame']).size().unstack(fill_value=0).to_string())
print('\n=== CONTENT — argument reasoning_type by creator ===')
print(con_out.groupby(['creator', 'argument_mining__reasoning_type']).size().unstack(fill_value=0).to_string())


=== AUDIENCE — sentiment by creator ===
sentiment__sentiment  mixed  negative  neutral  positive
creator                                                 
Agba John Doe            24        77       36        24
Banky Wellington         14        21       28        47
Deyemi Okanlawon         12        51       12         5
Shola                    12        32       12        10

=== AUDIENCE — sentiment by orientation ===
sentiment__sentiment  mixed  negative  neutral  positive
orientation                                             
progressive              26        72       40        52
regressive               36       109       48        34

=== AUDIENCE — top themes (controlled vocabulary) ===
themes__themes
male_sexual_entitlement     137
female_blame                123
male_accountability         113
male_authority_provider      92
female_submission_role       81
marriage_relationships       73
sexual_violence              53
male_emotional_life          39
feminism_gender_eq

## 6.5 — Sentiment remapped to 3-class

For comparability with the prior project's reporting (Pos/Neu/Neg only), the 4-class sentiment is collapsed: `mixed` → `neutral`. The 4-class column is preserved.

In [8]:
def to_3class(s):
    if s == 'mixed': return 'neutral'
    if s in ('positive', 'neutral', 'negative'): return s
    return 'neutral'

aud_out['sentiment__sentiment_3class'] = aud_out['sentiment__sentiment'].map(to_3class)
con_out['sentiment__sentiment_3class'] = con_out['sentiment__sentiment'].map(to_3class)

print('AUDIENCE — 3-class sentiment by creator')
print(aud_out.groupby(['creator', 'sentiment__sentiment_3class']).size().unstack(fill_value=0).to_string())
print('\nCONTENT — 3-class sentiment by creator')
print(con_out.groupby(['creator', 'sentiment__sentiment_3class']).size().unstack(fill_value=0).to_string())

AUDIENCE — 3-class sentiment by creator
sentiment__sentiment_3class  negative  neutral  positive
creator                                                 
Agba John Doe                      77       60        24
Banky Wellington                   21       42        47
Deyemi Okanlawon                   51       24         5
Shola                              32       24        10

CONTENT — 3-class sentiment by creator
sentiment__sentiment_3class  negative  neutral  positive
creator                                                 
Agba John Doe                      28       16         8
Banky Wellington                   15       33        28
Deyemi Okanlawon                    8        7         9
Ebuka Obi-Uchendu                   4       25         8
Shola                              50       12        10
Wizarab                            29       17         3


## 6.6 — Theme × sentiment / emotion heatmaps

Replicates the heatmap-style visualisation from the prior project (Appendix G). For each dataset, plots:
- **Theme × sentiment** — % of each theme that is positive / neutral / negative
- **Theme × emotion** — % of each theme that maps to each emotion

Heatmaps are saved as PNG files in the same folder as the workbook.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

OUT_FIG_DIR = OUT_AUDIENCE_DIR / 'figures'
OUT_FIG_DIR.mkdir(exist_ok=True)
OUT_FIG_DIR_C = OUT_CONTENT_DIR / 'figures'
OUT_FIG_DIR_C.mkdir(exist_ok=True)

def explode_listcol(df, col):
    s = df[col].apply(lambda x: x if isinstance(x, list) else ([] if x is None or (isinstance(x, float) and pd.isna(x)) else [x]))
    return df.assign(**{col: s}).explode(col)

def theme_x_label_heatmap(df, theme_col, label_col, name, out_dir, normalize='row'):
    """Plot %-share heatmap of theme × label."""
    ex = explode_listcol(df, theme_col)
    ex = ex.dropna(subset=[theme_col, label_col])
    if ex.empty:
        print(f'  skip {name} (no data)'); return None
    ct = pd.crosstab(ex[theme_col], ex[label_col])
    if normalize == 'row':
        pct = ct.div(ct.sum(axis=1), axis=0) * 100
    else:
        pct = ct
    # keep top 15 themes by frequency for readability
    ct['__total'] = ct.sum(axis=1)
    keep = ct.sort_values('__total', ascending=False).head(15).index
    pct = pct.loc[keep]
    fig, ax = plt.subplots(figsize=(max(6, len(pct.columns) * 0.9), max(4, len(pct.index) * 0.45)))
    sns.heatmap(pct, annot=True, fmt='.0f', cmap='RdYlGn_r' if 'sent' in label_col else 'YlOrRd', cbar_kws={'label': '% of theme'}, ax=ax)
    ax.set_title(f'{name}\nrow %-share (each row sums to 100%)')
    ax.set_xlabel(label_col.replace('__', '.'))
    ax.set_ylabel(theme_col.replace('__', '.'))
    plt.tight_layout()
    p = out_dir / f'{name}.png'
    fig.savefig(p, dpi=140, bbox_inches='tight')
    plt.close(fig)
    print(f'  wrote {p.name}  ({pct.shape[0]} themes × {pct.shape[1]} labels)')
    return pct

# AUDIENCE
print('AUDIENCE heatmaps')
theme_x_label_heatmap(aud_out, 'themes__themes', 'sentiment__sentiment_3class', 'audience_theme_x_sentiment', OUT_FIG_DIR)
# explode emotions list before crosstab
aud_em = explode_listcol(aud_out, 'emotion__emotions').rename(columns={'emotion__emotions': 'emotion'})
theme_x_label_heatmap(aud_em, 'themes__themes', 'emotion', 'audience_theme_x_emotion', OUT_FIG_DIR)
# stance variant
theme_x_label_heatmap(aud_out, 'themes__themes', 'stance__stance', 'audience_theme_x_stance', OUT_FIG_DIR)

# CONTENT
print('\nCONTENT heatmaps')
theme_x_label_heatmap(con_out, 'themes__themes', 'sentiment__sentiment_3class', 'content_theme_x_sentiment', OUT_FIG_DIR_C)
con_em = explode_listcol(con_out, 'emotion__emotions').rename(columns={'emotion__emotions': 'emotion'})
theme_x_label_heatmap(con_em, 'themes__themes', 'emotion', 'content_theme_x_emotion', OUT_FIG_DIR_C)
# framing variant
theme_x_label_heatmap(con_out, 'themes__themes', 'framing__frame', 'content_theme_x_frame', OUT_FIG_DIR_C)


## 6.7 — Theme co-occurrence matrices

For each dataset, count how often each pair of themes co-occurs in the same comment / segment. Replicates the co-occurrence matrix in the prior project's Appendix G.

In [10]:
def cooccurrence_matrix(df, theme_col, name, out_dir):
    """Build symmetric theme x theme co-occurrence count matrix."""
    rows = df[theme_col].apply(lambda x: x if isinstance(x, list) else [])
    rows = [list(set([t for t in r if t])) for r in rows]
    themes = sorted({t for r in rows for t in r})
    if not themes:
        print(f'  skip {name} (no themes)'); return None
    idx = {t: i for i, t in enumerate(themes)}
    M = np.zeros((len(themes), len(themes)), dtype=int)
    for r in rows:
        for i, a in enumerate(r):
            for b in r[i:]:
                M[idx[a], idx[b]] += 1
                if a != b:
                    M[idx[b], idx[a]] += 1
    co = pd.DataFrame(M, index=themes, columns=themes)
    # heatmap
    fig, ax = plt.subplots(figsize=(max(8, len(themes) * 0.55), max(6, len(themes) * 0.5)))
    sns.heatmap(co, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'co-occurrences'})
    ax.set_title(f'{name} — theme co-occurrence (diagonal = total occurrences)')
    plt.tight_layout()
    p = out_dir / f'{name}.png'
    fig.savefig(p, dpi=140, bbox_inches='tight')
    plt.close(fig)
    print(f'  wrote {p.name}  ({len(themes)} × {len(themes)})')
    return co

print('AUDIENCE co-occurrence')
aud_cooc = cooccurrence_matrix(aud_out, 'themes__themes', 'audience_theme_cooccurrence', OUT_FIG_DIR)
print('\nCONTENT co-occurrence')
con_cooc = cooccurrence_matrix(con_out, 'themes__themes', 'content_theme_cooccurrence', OUT_FIG_DIR_C)

# Save matrices as CSV alongside PNG
if aud_cooc is not None:
    aud_cooc.to_csv(OUT_FIG_DIR / 'audience_theme_cooccurrence.csv')
if con_cooc is not None:
    con_cooc.to_csv(OUT_FIG_DIR_C / 'content_theme_cooccurrence.csv')


AUDIENCE co-occurrence


  wrote audience_theme_cooccurrence.png  (14 × 14)

CONTENT co-occurrence


  wrote content_theme_cooccurrence.png  (15 × 15)


## 6.8 — Theme × sentiment heatmaps split by orientation

Helps show whether progressive vs regressive creator audiences differ in *which themes* skew negative / positive.

In [11]:
def split_heatmaps(df, name, out_dir):
    for orient in sorted(df['orientation'].dropna().unique()):
        sub = df[df['orientation'] == orient]
        theme_x_label_heatmap(sub, 'themes__themes', 'sentiment__sentiment_3class', f'{name}_{orient}_theme_x_sentiment', out_dir)

print('AUDIENCE — split by orientation')
split_heatmaps(aud_out, 'audience', OUT_FIG_DIR)
print('\nCONTENT — split by orientation')
split_heatmaps(con_out, 'content', OUT_FIG_DIR_C)


AUDIENCE — split by orientation
  wrote audience_progressive_theme_x_sentiment.png  (14 themes × 3 labels)


  wrote audience_regressive_theme_x_sentiment.png  (13 themes × 3 labels)

CONTENT — split by orientation


  wrote content_progressive_theme_x_sentiment.png  (13 themes × 3 labels)


  wrote content_regressive_theme_x_sentiment.png  (14 themes × 3 labels)


## 7 — Export to manager-facing workbooks

Two workbooks, each with **exactly two tabs**:

- **`Kenya - Audience LLM Exploratory Data Analyses.xlsx`**
  - `Summary` — counts and methodology
  - `audience` — `comment_id`, `source_url`, `comment` + every analysis column

- **`Kenya - Content LLM Exploratory Data Analyses.xlsx`**
  - `Summary` — counts and methodology
  - `content` — `content_id`, `context`, `verbatim_text` + every analysis column


In [12]:
def listify_for_excel(df):
    out = df.copy()
    for c in out.columns:
        if out[c].apply(lambda x: isinstance(x, (list, dict))).any():
            out[c] = out[c].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)
    return out

def analysis_cols(df, exclude):
    return [c for c in df.columns if c not in exclude]

# ── audience export ────────────────────────────────────────────────────────
aud_keep_cols = ['comment_id', 'source_url', 'creator', 'platform', 'orientation', 'language_detected', 'text_original', 'text_english']
aud_analysis_cols = [c for c in aud_out.columns if '__' in c or c.startswith('topic_cluster')]
aud_export = aud_out[aud_keep_cols + aud_analysis_cols].copy()
aud_export = aud_export.rename(columns={'text_english': 'comment', 'text_original': 'comment_original'})
# place core 3 first as Ksenia requested
first = ['comment_id', 'source_url', 'comment']
aud_export = aud_export[first + [c for c in aud_export.columns if c not in first]]

aud_summary_rows = [
    (f'{COUNTRY} — Audience LLM Exploratory Data Analyses', '', '', '', ''),
    ('Norman Lear Center × Gates Foundation', '', '', '', ''),
    ('', '', '', '', ''),
    ('Total comments', len(aud_export), '', '', ''),
    ('Creators', aud_export['creator'].nunique(), '', '', ''),
    ('Source workbook', f'{COUNTRY} Audience Analysis Final.xlsx', '', '', ''),
    ('Translation model', 'gpt-4o-mini', '', '', ''),
    ('Analysis model', 'gpt-4o-mini', '', '', ''),
    ('Embedding model', 'text-embedding-3-small (UMAP + HDBSCAN clustering)', '', '', ''),
    ('', '', '', '', ''),
    ('Comments by creator', '', '', '', ''),
]
for cr, n in aud_export['creator'].value_counts().items():
    aud_summary_rows.append((cr, n, '', '', ''))
aud_summary_rows.append(('', '', '', '', ''))
aud_summary_rows.append(('Sentiment distribution', '', '', '', ''))
for v, n in aud_export['sentiment__sentiment'].value_counts().items():
    aud_summary_rows.append((str(v), n, '', '', ''))
aud_summary_rows.append(('', '', '', '', ''))
aud_summary_rows.append(('Stance distribution', '', '', '', ''))
for v, n in aud_export['stance__stance'].value_counts().items():
    aud_summary_rows.append((str(v), n, '', '', ''))
aud_summary_rows.append(('', '', '', '', ''))
aud_summary_rows.append(('Top emergent topic clusters', '', '', '', ''))
for (_, lbl), n in aud_export.groupby(['topic_cluster_id', 'topic_cluster_label']).size().sort_values(ascending=False).head(10).items():
    aud_summary_rows.append((str(lbl), n, '', '', ''))
aud_summary_df = pd.DataFrame(aud_summary_rows, columns=['', '', '', '', ''])

with pd.ExcelWriter(OUT_AUDIENCE_XLSX, engine='openpyxl') as xw:
    aud_summary_df.to_excel(xw, sheet_name='Summary', index=False, header=False)
    listify_for_excel(aud_export).to_excel(xw, sheet_name='audience', index=False)
aud_out.to_parquet(OUT_AUDIENCE_PQ, index=False)
print(f'wrote {OUT_AUDIENCE_XLSX}')
print(f'wrote {OUT_AUDIENCE_PQ}')

# ── content export ─────────────────────────────────────────────────────────
con_keep_cols = ['content_id', 'context', 'creator', 'platform', 'content_type', 'orientation', 'source_url', 'language_detected', 'text_original', 'text_english']
con_analysis_cols = [c for c in con_out.columns if '__' in c or c.startswith('topic_cluster')]
con_export = con_out[con_keep_cols + con_analysis_cols].copy()
con_export = con_export.rename(columns={'text_english': 'verbatim_text', 'text_original': 'verbatim_text_original'})
first = ['content_id', 'context', 'verbatim_text']
con_export = con_export[first + [c for c in con_export.columns if c not in first]]

con_summary_rows = [
    (f'{COUNTRY} — Content LLM Exploratory Data Analyses', '', '', '', ''),
    ('Norman Lear Center × Gates Foundation', '', '', '', ''),
    ('', '', '', '', ''),
    ('Total content segments', len(con_export), '', '', ''),
    ('Creators', con_export['creator'].nunique(), '', '', ''),
    ('Source workbook', f'{COUNTRY} Content Analysis Final.xlsx', '', '', ''),
    ('Translation model', 'gpt-4o-mini', '', '', ''),
    ('Analysis model', 'gpt-4o-mini', '', '', ''),
    ('Embedding model', 'text-embedding-3-small (UMAP + HDBSCAN clustering)', '', '', ''),
    ('', '', '', '', ''),
    ('Segments by creator', '', '', '', ''),
]
for cr, n in con_export['creator'].value_counts().items():
    con_summary_rows.append((cr, n, '', '', ''))
con_summary_rows.append(('', '', '', '', ''))
con_summary_rows.append(('Sentiment distribution', '', '', '', ''))
for v, n in con_export['sentiment__sentiment'].value_counts().items():
    con_summary_rows.append((str(v), n, '', '', ''))
con_summary_rows.append(('', '', '', '', ''))
con_summary_rows.append(('Frame distribution', '', '', '', ''))
for v, n in con_export['framing__frame'].value_counts().items():
    con_summary_rows.append((str(v), n, '', '', ''))
con_summary_rows.append(('', '', '', '', ''))
con_summary_rows.append(('Top emergent topic clusters', '', '', '', ''))
for (_, lbl), n in con_export.groupby(['topic_cluster_id', 'topic_cluster_label']).size().sort_values(ascending=False).head(10).items():
    con_summary_rows.append((str(lbl), n, '', '', ''))
con_summary_df = pd.DataFrame(con_summary_rows, columns=['', '', '', '', ''])

with pd.ExcelWriter(OUT_CONTENT_XLSX, engine='openpyxl') as xw:
    con_summary_df.to_excel(xw, sheet_name='Summary', index=False, header=False)
    listify_for_excel(con_export).to_excel(xw, sheet_name='content', index=False)
con_out.to_parquet(OUT_CONTENT_PQ, index=False)
print(f'wrote {OUT_CONTENT_XLSX}')
print(f'wrote {OUT_CONTENT_PQ}')


wrote /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project/Nigeria/Audience Analysis/Nigeria - Audience LLM Exploratory Data Analyses.xlsx
wrote /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project/Nigeria/Audience Analysis/audience_exploratory_results.parquet
wrote /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project/Nigeria/Content Analysis/Nigeria - Content LLM Exploratory Data Analyses.xlsx
wrote /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project/Nigeria/Content Analysis/content_exploratory_results.parquet


## Notes

- All LLM calls cache to `temp/exploratory_cache/<analysis>.parquet`. Re-running the notebook is free for already-processed text.
- Embedding cache lives at `temp/exploratory_cache/embeddings_<dataset>.parquet`.
- To force a single analysis to re-run, delete its cache file in `temp/exploratory_cache/`.
- Estimated cost (gpt-4o-mini + text-embedding-3-small) depends on final Kenya row counts, but should remain low for this project-scale run. Wall time is usually ~5–10 min for a similar project-scale run.
- Stance is run on audience only (Ksenia: "more relevant for audience reception"). Framing + argument mining are run on content only (Ksenia: "all the other ones are very relevant for content").
- Kenya orientation labels follow the final country memo: Amerix and Andrew Kibe as regressive; Philip Karanja, Onyango Otieno, and Eddy Kimani as progressive / vulnerability-forward.
